In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("/Users/jeannegautier/Projets/AI-projet/data/base-etablissements.csv")

/var/folders/bm/4ts3_qzd727_k6ls8w3b5kbc0000gn/T/ipykernel_18577/3695786910.py:1: DtypeWarning: Columns (0: noFinesset, 1: coordinates.deptcode, 2: ehpadPrice._id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/jeannegautier/Projets/AI-projet/data/base-etablissements.csv")


In [4]:
df.shape
df.info()
df.head()
df.columns

<class 'pandas.DataFrame'>
RangeIndex: 10901 entries, 0 to 10900
Data columns (total 70 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   _id                         10901 non-null  int64  
 1   title                       10901 non-null  str    
 2   updatedAt                   10896 non-null  str    
 3   noFinesset                  10901 non-null  object 
 4   capacity                    10901 non-null  int64  
 5   legal_status                10894 non-null  str    
 6   isViaTrajectoire            0 non-null      float64
 7   IsEHPAD                     10901 non-null  bool   
 8   IsEHPA                      10901 non-null  bool   
 9   IsESLD                      10901 non-null  bool   
 10  IsRA                        10901 non-null  bool   
 11  IsAJA                       10901 non-null  bool   
 12  IsHCOMPL                    10901 non-null  bool   
 13  IsHTEMPO                    10901 non-null

Index(['_id', 'title', 'updatedAt', 'noFinesset', 'capacity', 'legal_status',
       'isViaTrajectoire', 'IsEHPAD', 'IsEHPA', 'IsESLD', 'IsRA', 'IsAJA',
       'IsHCOMPL', 'IsHTEMPO', 'IsACC_JOUR', 'IsACC_NUIT', 'IsHAB_AIDE_SOC',
       'IsCONV_APL', 'IsALZH', 'IsUHR', 'IsPASA', 'IsPUV', 'IsF1', 'IsF1Bis',
       'IsF2', 'cerfa', 'prixMin', 'coordinates._id', 'coordinates.title',
       'coordinates.isPublished', 'coordinates.createdAt',
       'coordinates.updatedAt', 'coordinates.street', 'coordinates.postcode',
       'coordinates.deptcode', 'coordinates.deptname', 'coordinates.city',
       'coordinates.phone', 'coordinates.emailContact',
       'coordinates.gestionnaire', 'coordinates.website',
       'coordinates.latitude', 'coordinates.longitude', 'ehpadPrice._id',
       'ehpadPrice.updatedAt', 'ehpadPrice.prixHebPermCs',
       'ehpadPrice.prixHebPermCd', 'ehpadPrice.prixHebPermCsa',
       'ehpadPrice.prixHebPermCda', 'ehpadPrice.prixHebTempCs',
       'ehpadPrice.prixHebTemp

In [5]:
missing = df.isna().mean().sort_values(ascending=False)
missing.head(30)

cerfa                         1.000000
coordinates._id               1.000000
coordinates.updatedAt         1.000000
coordinates.createdAt         1.000000
coordinates.title             1.000000
isViaTrajectoire              1.000000
raPrice                       1.000000
ehpadPrice.tarifHebJour       1.000000
coordinates.isPublished       1.000000
raPrice.PrixF2ASH             0.975415
raPrice.PrixF1ASH             0.966609
raPrice.PrixF1BisASH          0.965233
ehpadPrice.prixHebTempCda     0.924411
raPrice.autreTarifPrest       0.923126
raPrice.PrixF1                0.905330
raPrice.PrixF2                0.887442
ehpadPrice.prixHebTempCd      0.877534
raPrice.PrixF1Bis             0.866159
raPrice.prestObligatoire      0.831208
raPrice._id                   0.821576
raPrice.updatedAt             0.821576
ehpadPrice.prixHebTempCsa     0.812311
ehpadPrice.prixHebPermCda     0.733786
ehpadPrice.prixHebTempCs      0.673883
ehpadPrice.autreTarifPrest    0.664434
ehpadPrice.prixHebPermCd 

In [7]:
df["noFinesset"].duplicated().sum()
df["_id"].duplicated().sum()

np.int64(0)

In [8]:
date_cols = [
    "updatedAt",
    "ehpadPrice.updatedAt",
    "raPrice.updatedAt"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

In [9]:
df["noFinesset"] = df["noFinesset"].astype(str) #could start by 0 so better str than int

In [10]:
empty_cols = df.columns[df.isna().mean() == 1]
df = df.drop(columns=empty_cols)

In [13]:
ehpad = df[df["IsEHPAD"] == True]

ehpad_missing_price = ehpad["ehpadPrice.prixHebPermCs"].isna().mean()
print(ehpad_missing_price)

df["ehpadPrice.prixHebPermCs"].isna().mean() 

0.030460921843687375


np.float64(0.3326300339418402)

In [14]:
ra = df[df["IsRA"] == True]

ra_missing_price = ra[
    ["raPrice.PrixF1", "raPrice.PrixF1Bis", "raPrice.PrixF2"]
].isna().mean()

ra_missing_price

raPrice.PrixF1       0.547336
raPrice.PrixF1Bis    0.359313
raPrice.PrixF2       0.461471
dtype: float64

In [15]:
diagnostic = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "missing": df.isna().sum().values,
    "missing_rate": df.isna().mean().values,
    "n_unique": df.nunique(dropna=True).values
})

diagnostic = diagnostic.sort_values("missing_rate", ascending=False)
diagnostic.head(30)

,column,dtype,non_null,missing,missing_rate,n_unique
58,raPrice.PrixF2ASH,float64,268,10633,0.975415,245
54,raPrice.PrixF1ASH,float64,364,10537,0.966609,338
56,raPrice.PrixF1BisASH,float64,379,10522,0.965233,353
45,ehpadPrice.prixHebTempCda,float64,824,10077,0.924411,646
59,raPrice.autreTarifPrest,str,838,10063,0.923126,798
53,raPrice.PrixF1,float64,1032,9869,0.905330,965
57,raPrice.PrixF2,float64,1227,9674,0.887442,1102
43,ehpadPrice.prixHebTempCd,float64,1335,9566,0.877534,934
55,raPrice.PrixF1Bis,float64,1459,9442,0.866159,1275
60,raPrice.prestObligatoire,str,1840,9061,0.831208,541


In [4]:
print(df.columns.tolist())

['_id', 'title', 'updatedAt', 'noFinesset', 'capacity', 'legal_status', 'isViaTrajectoire', 'IsEHPAD', 'IsEHPA', 'IsESLD', 'IsRA', 'IsAJA', 'IsHCOMPL', 'IsHTEMPO', 'IsACC_JOUR', 'IsACC_NUIT', 'IsHAB_AIDE_SOC', 'IsCONV_APL', 'IsALZH', 'IsUHR', 'IsPASA', 'IsPUV', 'IsF1', 'IsF1Bis', 'IsF2', 'cerfa', 'prixMin', 'coordinates._id', 'coordinates.title', 'coordinates.isPublished', 'coordinates.createdAt', 'coordinates.updatedAt', 'coordinates.street', 'coordinates.postcode', 'coordinates.deptcode', 'coordinates.deptname', 'coordinates.city', 'coordinates.phone', 'coordinates.emailContact', 'coordinates.gestionnaire', 'coordinates.website', 'coordinates.latitude', 'coordinates.longitude', 'ehpadPrice._id', 'ehpadPrice.updatedAt', 'ehpadPrice.prixHebPermCs', 'ehpadPrice.prixHebPermCd', 'ehpadPrice.prixHebPermCsa', 'ehpadPrice.prixHebPermCda', 'ehpadPrice.prixHebTempCs', 'ehpadPrice.prixHebTempCd', 'ehpadPrice.prixHebTempCsa', 'ehpadPrice.prixHebTempCda', 'ehpadPrice.tarifHebJour', 'ehpadPrice.ta